In [3]:
!pip install -qq transformers peft decord scikit-learn tqdm

import os, json, random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.io import read_image
from transformers import TimesformerForVideoClassification
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm
import gc

# Updated AMP import with fallback
try:
    from torch.amp import autocast, GradScaler
    AMP_DEVICE = 'cuda'
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_DEVICE = None

# Set environment variable to avoid memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# === PATHS ===
DATASET_DIR = "/content/drive/MyDrive/SceneSolverG388/CoreInferEngine/BinaryAnomalyDetection"
ANOMALY_FRAMES_DIR = os.path.join(DATASET_DIR, "anomaly_frames")
NORMAL_FRAMES_DIR = os.path.join(DATASET_DIR, "normal_frames")

MODELS_DIR = "/content/drive/MyDrive/SceneSolverG388/CoreInferEngine/TimeSformerBinaryModels"
CKPT_DIR = os.path.join(MODELS_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(MODELS_DIR, "timesformer_best_binary.pth")
LATEST_CKPT_PATH = os.path.join(CKPT_DIR, "latest_checkpoint_binary.pth")
CONFIG_PATH = os.path.join(MODELS_DIR, "training_config_binary.json")

# === TRANSFORMS === (ImageNet normalization)
transform = T.Compose([
    T.Resize((224, 224)),
    T.ConvertImageDtype(torch.float32),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# === AUGMENTATION TRANSFORMS (training only) ===
augment_transform = T.Compose([
    T.Resize((224, 224)),
    T.ConvertImageDtype(torch.float32),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# === CUSTOM DATASET CLASS USING PRE-EXTRACTED FRAMES ===
class FrameFolderDataset(Dataset):
    def __init__(self, root_dir, label, frame_count, augment=False):
        self.label = label
        self.frame_count = frame_count
        self.augment = augment
        self.samples = []
        for video_folder in os.listdir(root_dir):
            full_path = os.path.join(root_dir, video_folder)
            if os.path.isdir(full_path):
                self.samples.append(full_path)

    def get_label(self):
        """Return the scalar label for all samples — no disk I/O."""
        return self.label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        folder = self.samples[idx]
        frames = sorted([f for f in os.listdir(folder) if f.endswith((".jpg", ".png"))])
        total_frames = len(frames)

        if total_frames >= self.frame_count:
            indices = torch.linspace(0, total_frames - 1, self.frame_count).long()
        else:
            indices = torch.cat([
                torch.arange(total_frames),
                torch.full((self.frame_count - total_frames,), total_frames - 1)
            ])

        tfm = augment_transform if self.augment else transform
        imgs = []
        for i in indices:
            img_path = os.path.join(folder, frames[i])
            img = read_image(img_path)
            img = tfm(img)
            imgs.append(img)

        video_tensor = torch.stack(imgs)  # [T, C, H, W]
        return video_tensor, torch.tensor(self.label)


# === COMBINED DATASET TO MIX NORMAL AND ANOMALY ===
class CombinedDataset(Dataset):
    def __init__(self, anomaly_ds, normal_ds):
        self.anomaly_ds = anomaly_ds
        self.normal_ds = normal_ds
        self.len_a = len(anomaly_ds)
        self.len_n = len(normal_ds)
        # Pre-build flat label list — O(n) integers, zero disk I/O
        self._labels = (
            [anomaly_ds.get_label()] * self.len_a +
            [normal_ds.get_label()] * self.len_n
        )

    def get_labels(self):
        """Return all labels instantly without loading any frames."""
        return self._labels

    def __len__(self):
        return self.len_a + self.len_n

    def __getitem__(self, idx):
        if idx < self.len_a:
            return self.anomaly_ds[idx]
        else:
            return self.normal_ds[idx - self.len_a]


# === MODEL SETUP ===
def get_model():
    model = TimesformerForVideoClassification.from_pretrained(
        "facebook/timesformer-base-finetuned-k400",
        num_labels=2,
        ignore_mismatched_sizes=True
    )
    return model


# === SPLIT DATASET ===
def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    n = len(dataset)
    train_size = int(train_ratio * n)
    val_size = int(val_ratio * n)
    test_size = n - train_size - val_size
    return random_split(dataset, [train_size, val_size, test_size])


# === WEIGHTED SAMPLER FOR CLASS BALANCE ===
def make_weighted_sampler(subset):
    """
    Fast weighted sampler — reads labels from in-memory list, no disk I/O.
    Uses the pre-built _labels list from CombinedDataset via subset.indices.
    """
    all_labels = subset.dataset.get_labels()          # instant, in-memory
    labels = [all_labels[i] for i in subset.indices]  # index into list

    class_counts = [labels.count(c) for c in [0, 1]]
    class_weights = [1.0 / max(c, 1) for c in class_counts]
    sample_weights = [class_weights[l] for l in labels]

    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )


# === RECOVER BEST MODEL FROM LATEST CHECKPOINT ===
def recover_best_model_from_checkpoint(device):
    """
    If the best model file is missing but a latest checkpoint exists,
    restore model weights from the checkpoint as the new best model.
    """
    if not os.path.exists(BEST_MODEL_PATH) and os.path.exists(LATEST_CKPT_PATH):
        print("⚠️  Best model not found. Recovering weights from latest checkpoint...")
        checkpoint = torch.load(LATEST_CKPT_PATH, map_location=device)
        torch.save(checkpoint["model"], BEST_MODEL_PATH)
        recovered_acc = checkpoint.get("best_acc", 0.0)
        print(f"✅  Best model recovered from checkpoint (recorded acc: {recovered_acc:.4f})")
        return recovered_acc
    return None


# === TRAIN FUNCTION ===
def train(epochs=10, batch_size=1, grad_accum=8, patience=3):
    # --- Datasets ---
    anomaly_ds_train = FrameFolderDataset(ANOMALY_FRAMES_DIR, label=1, frame_count=32, augment=True)
    normal_ds_train  = FrameFolderDataset(NORMAL_FRAMES_DIR,  label=0, frame_count=32, augment=True)
    anomaly_ds_eval  = FrameFolderDataset(ANOMALY_FRAMES_DIR, label=1, frame_count=32, augment=False)
    normal_ds_eval   = FrameFolderDataset(NORMAL_FRAMES_DIR,  label=0, frame_count=32, augment=False)

    train_combined = CombinedDataset(anomaly_ds_train, normal_ds_train)
    eval_combined  = CombinedDataset(anomaly_ds_eval,  normal_ds_eval)

    train_ds, _, _         = split_dataset(train_combined)
    _, val_ds_eval, test_ds_eval = split_dataset(eval_combined)

    # Fast sampler — no frames loaded here
    print("Building weighted sampler...")
    sampler = make_weighted_sampler(train_ds)
    print("Sampler ready.")

    train_dl = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds_eval,  batch_size=batch_size, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds_eval, batch_size=batch_size, num_workers=2, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Attempt to recover best model if missing
    recover_best_model_from_checkpoint(device)

    model = get_model().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=5e-6, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    if AMP_DEVICE:
        scaler = GradScaler(AMP_DEVICE)
    else:
        scaler = GradScaler()

    start_epoch = 1
    best_acc = 0.0
    early_stop_counter = 0

    # --- Resume from checkpoint ---
    if os.path.exists(LATEST_CKPT_PATH):
        print("Loading checkpoint...")
        checkpoint = torch.load(LATEST_CKPT_PATH, map_location=device)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        if "scheduler" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch = checkpoint["epoch"] + 1
        best_acc = checkpoint["best_acc"]
        print(f"Resumed from epoch {checkpoint['epoch']} | Best acc so far: {best_acc:.4f}")

    # --- Training Loop ---
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        pbar = tqdm(train_dl, desc=f"Epoch {epoch}/{epochs}")
        for i, (x, y) in enumerate(pbar):
            x, y = x.to(device), y.to(device)

            if i == 0 and epoch == start_epoch:
                print(f"Input shape: {x.shape}")

            ctx = autocast(AMP_DEVICE) if AMP_DEVICE else autocast()
            with ctx:
                outputs = model(pixel_values=x, labels=y)
                loss = outputs.loss / grad_accum

            scaler.scale(loss).backward()
            running_loss += loss.item() * grad_accum

            if (i + 1) % grad_accum == 0 or (i + 1) == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                gc.collect()

            avg_loss = running_loss / (i + 1)
            pbar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch} | Train Loss: {running_loss / len(train_dl):.4f} | LR: {current_lr:.2e}")

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        correct, total = 0, 0
        preds, targs = [], []

        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                ctx = autocast(AMP_DEVICE) if AMP_DEVICE else autocast()
                with ctx:
                    outputs = model(pixel_values=x, labels=y)
                    pred = torch.argmax(outputs.logits, dim=1)
                    val_loss += outputs.loss.item()
                correct += (pred == y).sum().item()
                total += y.size(0)
                preds.extend(pred.cpu().tolist())
                targs.extend(y.cpu().tolist())

        val_acc = correct / total
        avg_val_loss = val_loss / len(val_dl)
        print(f"Validation Accuracy: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            early_stop_counter = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"✅  Best model saved (acc: {best_acc:.4f})")
        else:
            early_stop_counter += 1
            print(f"No improvement. Early stop counter: {early_stop_counter}/{patience}")
            if early_stop_counter >= patience:
                print("Early stopping triggered.")
                break

        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_acc": best_acc,
        }, LATEST_CKPT_PATH)

        with open(CONFIG_PATH, "w") as f:
            json.dump({
                "last_epoch": epoch,
                "best_acc": best_acc,
                "batch_size": batch_size,
                "grad_accum": grad_accum,
                "last_lr": current_lr,
            }, f, indent=2)

    evaluate(model, test_dl, device)


# === EVALUATE FUNCTION ===
def evaluate(model, dataloader, device):
    model.eval()
    preds, targs = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            ctx = autocast(AMP_DEVICE) if AMP_DEVICE else autocast()
            with ctx:
                outputs = model(pixel_values=x)
                pred = torch.argmax(outputs.logits, dim=1)
            preds.extend(pred.cpu().tolist())
            targs.extend(y.cpu().tolist())

    print("\n=== Test Evaluation ===")
    print(f"Test Accuracy: {accuracy_score(targs, preds):.4f}")
    print("Classification Report:\n", classification_report(targs, preds, digits=4, target_names=["Normal", "Anomaly"]))


# === INFERENCE FUNCTION ===
def infer_single(video_folder_path, frame_count=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_model().to(device)

    if not os.path.exists(BEST_MODEL_PATH):
        recovered = recover_best_model_from_checkpoint(device)
        if recovered is None:
            raise FileNotFoundError(
                f"No best model found at {BEST_MODEL_PATH} and no checkpoint to recover from."
            )

    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    model.eval()

    frames_files = sorted([f for f in os.listdir(video_folder_path) if f.endswith((".jpg", ".png"))])
    total_frames = len(frames_files)

    if total_frames >= frame_count:
        indices = torch.linspace(0, total_frames - 1, frame_count).long()
    else:
        indices = torch.cat([
            torch.arange(total_frames),
            torch.full((frame_count - total_frames,), total_frames - 1)
        ])

    frames = []
    for i in indices:
        img_path = os.path.join(video_folder_path, frames_files[i])
        img = read_image(img_path)
        img = transform(img)
        frames.append(img)

    video_tensor = torch.stack(frames).unsqueeze(0).to(device)  # [1, T, C, H, W]

    ctx = autocast(AMP_DEVICE) if AMP_DEVICE else autocast()
    with torch.no_grad():
        with ctx:
            outputs = model(pixel_values=video_tensor)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            conf, pred = torch.max(probs, dim=1)

    label = "Anomaly" if pred.item() == 1 else "Normal"
    print(f"Prediction: {label} | Confidence: {conf.item():.4f}")
    return label, conf.item()


# === MAIN ===
if __name__ == "__main__":
    train()

Building weighted sampler...
Sampler ready.
Using device: cuda
⚠️  Best model not found. Recovering weights from latest checkpoint...
✅  Best model recovered from checkpoint (recorded acc: 0.9556)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

TimesformerForVideoClassification LOAD REPORT from: facebook/timesformer-base-finetuned-k400
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Loading checkpoint...


model.safetensors:   0%|          | 0.00/486M [00:00<?, ?B/s]

Resumed from epoch 13 | Best acc so far: 0.9556

=== Test Evaluation ===
Test Accuracy: 0.9667
Classification Report:
               precision    recall  f1-score   support

      Normal     1.0000    0.9231    0.9600        13
     Anomaly     0.9444    1.0000    0.9714        17

    accuracy                         0.9667        30
   macro avg     0.9722    0.9615    0.9657        30
weighted avg     0.9685    0.9667    0.9665        30

